# **Superposition and SAEs: practice**

This practice has two parts.

1. **Superposition in a toy model.** We repeat the experiment of [Elhage et al., 2022](https://transformer-circuits.pub/2022/toy_model/index.html): five features are compressed into two dimensions, and the sparsity of the features shows when the model starts storing them in superposition.
2. **SAE Lens.** We load a pretrained sparse autoencoder for GPT-2 small, look at how it is built, and read the features it finds in the residual stream.

The tasks are numbered; you enter the answers on the course website, in the lesson «Superposition and SAEs: Practice».

## **Set up**
A CPU runtime is enough for this practice: no extra compute is needed.

In [ ]:
!pip install transformer-lens==3.9.0 plotly -q

In [ ]:
!pip install sae-lens==6.54.0 -q

****
**After running the cells above, restart the notebook before starting the tasks.**
****

In [ ]:
# Standard libraries
import math
import torch
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

torch.set_grad_enabled(False)  # gradients are needed only to train the toy model; they are switched on there explicitly

# MPS is not chosen: TransformerLens warns that results on it may be wrong
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# **Part 1. Superposition in a toy model**

The model has $n = 5$ features and only $m = 2$ dimensions. With probability $p$ feature $x_i$ is a random number from $[0, 1]$, otherwise it is zero; $1 - p$ is called the sparsity. The model compresses the input with a matrix $W \in \mathbb{R}^{2 \times 5}$ and reconstructs it:

$$\hat x = \mathrm{ReLU}(W^\top W x + b).$$

The reconstruction error is weighted by feature importance $I_i = 0.9^i$: feature 0 is the most important. Column $W_i$ is the direction in which the model writes feature $i$ into the two-dimensional space. If the column length is close to 1, the feature is represented; if it is close to 0, the model has dropped it.

In [ ]:
N_FEATURES, N_DIMS = 5, 2
importance = 0.9 ** torch.arange(N_FEATURES)  # the importance of feature i is 0.9^i


def train_toy(p, steps=5000, seed=0):
    """Trains W (2 x 5) and b so that ReLU(W^T W x + b) reconstructs x.

    p is the probability that a feature is non-zero; 1 - p is the sparsity.
    The model is tiny and runs on the CPU, so the same seed gives the same result on any machine.
    """
    g = torch.Generator().manual_seed(seed)
    W = torch.nn.Parameter(torch.randn(N_DIMS, N_FEATURES, generator=g) * 0.1)
    b = torch.nn.Parameter(torch.zeros(N_FEATURES))
    opt = torch.optim.AdamW([W, b], lr=1e-2, weight_decay=0)
    with torch.enable_grad():
        for _ in range(steps):
            x = torch.rand(1024, N_FEATURES, generator=g) * (torch.rand(1024, N_FEATURES, generator=g) < p)
            x_hat = torch.relu(x @ W.T @ W + b)
            loss = (importance * (x - x_hat) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    return W.detach(), b.detach()

Let us train the model at five values of $p$, from dense features ($p = 1$) to very sparse ones ($p = 0.01$). For each $p$, count how many features are represented (column length above 0.5) and how many dimensions one feature gets: $m / \|W\|_F^2$. If there are as many features as dimensions and each takes its own, this number is 1; the smaller it is, the more tightly the features are packed.

In [ ]:
toy = {p: train_toy(p) for p in [1.0, 0.3, 0.1, 0.05, 0.01]}

for p, (W, b) in toy.items():
    norms = W.norm(dim=0)  # the length of column W_i: how strongly feature i is represented in the two dimensions
    represented = ...  # Your code here: the number of features whose column is longer than 0.5
    dims_per_feature = ...  # Your code here: N_DIMS / ||W||_F^2
    print(f"p={p}: lengths {[round(v, 2) for v in norms.tolist()]}, represented {represented}, dimensions per feature {dims_per_feature}")

**Task 1. How many features are represented with dense features, $p = 1$?**

**Task 2. How many features are represented at $p = 0.05$?**

**Task 3. How many dimensions does one feature get at $p = 0.05$? Round the answer to two decimal places.**

Let us draw the columns of $W$ for the dense and the sparse case and look at the angles between them and at the biases $b$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, p in zip(axes, [1.0, 0.05]):
    W, b = toy[p]
    for i in range(N_FEATURES):
        ax.plot([0, W[0, i]], [0, W[1, i]], marker="o")
        ax.annotate(f"{i}", (W[0, i] * 1.1, W[1, i] * 1.1))
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect("equal")
    ax.set_title(f"p = {p}")
plt.show()

W, b = toy[0.05]
angles = sorted((torch.atan2(W[1], W[0]) * 180 / math.pi).tolist())
print("column angles, degrees:", [round(a, 1) for a in angles])
print("biases b at p=0.05:", [round(v, 3) for v in b.tolist()])
print("biases b at p=1.0:", [round(v, 3) for v in toy[1.0][1].tolist()])

**Task 4. What is the sign of the biases $b$ at $p = 0.05$, and why does the model need that sign?**

At $p = 0.05$ five features lie in two dimensions, so the directions are not orthogonal: an active feature gives a small projection onto the other directions. This is superposition interference. Think about what ReLU together with the bias does to that interference.

# **Part 2. SAE Lens**

In the toy model we knew the features in advance. In a real model nobody knows them, and a sparse autoencoder (SAE) tries to find them itself: it decomposes an activation over a dictionary with more directions than dimensions. The [SAE Lens](https://github.com/decoderesearch/SAELens) library keeps a directory of trained SAEs for GPT-2, Gemma 2 (the [Gemma Scope](https://huggingface.co/google/gemma-scope) project), Llama and other models.

In [ ]:
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory

df = pd.DataFrame.from_records(
    {k: v.__dict__ for k, v in get_pretrained_saes_directory().items()}
).T.reset_index(drop=True)

df.head()

The table is organised as follows:

- `release` — a release, that is, a set of SAEs trained with one approach. For example, `gemma-scope-2b-pt-att`: `gemma-scope` is the Gemma Scope project, `2b-pt` is the base Gemma 2 2B model, `att` means the SAEs were trained on attention-layer outputs;
- `repo_id` — the Hugging Face repository with the weights; it often has a detailed card;
- `model` — the model on whose activations the SAEs were trained;
- `saes_map` — all SAEs of the release: the SAE name and the hook it attaches to;
- `neuronpedia_id` — filled in if the SAE features can be browsed on Neuronpedia.

Let us print the SAEs of two GPT-2 small releases: one SAE on the residual stream of each layer, and a release where SAEs of different sizes are trained on one hook.

In [ ]:
for release in ["gpt2-small-res-jb", "gpt2-small-res-jb-feature-splitting"]:
    print(f"SAEs in release {release}:")
    for sae_id, hook in df.loc[df.release == release, "saes_map"].values[0].items():
        print(f"  {sae_id} -> {hook}")

n_saes = ...  # Your code here: the number of SAEs in release gpt2-small-res-jb (the length of its saes_map)
print("SAEs in gpt2-small-res-jb:", n_saes)
print("Gemma Scope releases in the directory:", int(df.release.str.startswith("gemma-scope").sum()))

**Task 5. How many SAEs are in release `gpt2-small-res-jb`?**

**Task 6. How many different SAEs can be trained on one model?** Compare the number of SAEs in the two releases above with the number of layers in GPT-2 small (there are 12).

## **An SAE and its model**

Let us load GPT-2 small wrapped in TransformerLens and the SAE trained on the residual stream before block 7 (`blocks.7.hook_resid_pre`). Recall the points the residual stream passes through inside a block:

`x_pre ──► Attention ──► x_mid ──► MLP ──► x_post (= x_pre of the next block)`

In [ ]:
from sae_lens import SAE, HookedSAETransformer

model = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # release
    sae_id="blocks.7.hook_resid_pre",  # SAE id
    device=device,
)

In [ ]:
model.blocks[7]

The name of a hook tells you which point of the block it records.

**Task 7. Match each hook to its point in the model:** `hook_normalized`, `hook_q_input`, `hook_resid_mid`, `hook_resid_post`.

## **Inside the SAE**

An SAE is a model with hooks too. Its config stores the input dimension `d_in` and the dictionary size `d_sae`.

In [ ]:
print(sae)
print(sae.cfg.to_dict())

expansion = ...  # Your code here: the expansion factor d_sae / d_in
print("expansion factor:", expansion)

**Task 8. What is the expansion factor of this SAE?**

## **SAE features on Neuronpedia**

An SAE feature is one coordinate of the hidden vector, that is, one dictionary direction. What it means is decided from the texts on which it activates most strongly. [Neuronpedia](https://www.neuronpedia.org/) shows such examples for each feature, plots of the logits the feature affects, and a description written by a language model. The cell below shows the dashboard of feature 3777 (it works if the environment can open an external site).

In [ ]:
from IPython.display import IFrame

html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"


def get_dashboard_html(sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)


IFrame(get_dashboard_html(feature_idx=3777), width=1200, height=600)

Feature descriptions can be downloaded as a table and searched for a topic. Let us load the descriptions of our SAE's features. They are stored in the open Neuronpedia S3 dataset (several `jsonl.gz` archives); every request has a 60 s timeout and two retries, so without a network the cell stops with an error instead of hanging. In that case rerun the cell once the network is back.

In [ ]:
import gzip
import json
import re

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retries on network failures and a 60 s timeout per request, so the cell does not hang
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(total=2, backoff_factor=2)))


# Neuronpedia feature explanations are exported to the open S3 dataset neuronpedia-datasets (jsonl.gz batches)
def load_explanations(model_id, sae_id):
    base = "https://neuronpedia-datasets.s3.us-east-1.amazonaws.com"
    listing = session.get(base, params={"list-type": "2", "prefix": f"v1/{model_id}/{sae_id}/explanations/"}, timeout=60).text
    rows = []
    for key in re.findall(r"<Key>([^<]+\.jsonl\.gz)</Key>", listing):
        raw = gzip.decompress(session.get(f"{base}/{key}", timeout=60).content).decode()
        rows += [{k: v for k, v in json.loads(line).items() if k != "embedding"} for line in raw.splitlines()]
    return rows


explanations_df = pd.DataFrame(load_explanations("gpt2-small", "7-res-jb"))
explanations_df.rename(columns={"index": "feature"}, inplace=True)
explanations_df["description"] = explanations_df["description"].str.lower()
explanations_df[["feature", "description", "explanationModelName"]].head()

In [ ]:
fruits_features = ...  # Your code here: the rows of explanations_df whose description contains " fruits"
fruits_features

**Task 9. How many features of this SAE are described with the word «fruits»?**

The descriptions were written by an annotator model, not by a person, and a feature with a fitting description does not have to fire on every fruit: a description is a hypothesis that is checked on new texts.

## **Running the model with an SAE**

Let us see how the model continues a prompt, then run it together with the SAE and store all hooks in a cache.

In [ ]:
from transformer_lens.utils import test_prompt

prompt = "The best animal of the world is a"
answer = "cat"
test_prompt(prompt, answer, model)

SAE hooks:

- `hook_sae_input` — the SAE input, the model activation, `[batch, seq_len, d_in]`;
- `hook_sae_acts_pre` — $xW_{enc} + b_{enc}$ before ReLU, `[batch, seq_len, d_sae]`;
- `hook_sae_acts_post` — the features after ReLU, the ones that are read, `[batch, seq_len, d_sae]`;
- `hook_sae_recons` — the reconstruction $f W_{dec} + b_{dec}$, `[batch, seq_len, d_in]`;
- `hook_sae_error` — the reconstruction error; it is computed only if `sae.use_error_term = True`;
- `hook_sae_output` — what the SAE passes on through the model.

By default `use_error_term = False`: the reconstruction, not the original activation, goes on through the model.

In [ ]:
_, cache = model.run_with_cache_with_saes(prompt, saes=[sae])

# Your code here: print the names and shapes of all cache entries whose name contains "sae"

**Task 10. How many cache entries belong to the SAE?**

In [ ]:
acts = cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0]  # [position, d_sae]

px.line(acts[-1].cpu().numpy(), title="Feature activations at the final token position",
        labels={"index": "Feature", "value": "Activation"}).show()

vals, inds = torch.topk(acts[-1], 5)
for val, ind in zip(vals, inds):
    print(f"Feature {int(ind)} fired {val:.2f}")

**Task 11. What is the index of the feature with the largest activation at the last token?**

None of the five strongest features is about cats, although the model ranks « cat» third. Recall the lesson on the logit lens: at layer 7 of 12 the model has not assembled the answer yet.

In [ ]:
l0 = ...  # Your code here: the number of non-zero features at each prompt position
print("L0 by position:", l0)

**Task 12. What is $L_0$ at the last token of the prompt — how many of the 24,576 features are non-zero?**

In [ ]:
x = cache["blocks.7.hook_resid_pre.hook_sae_input"][0, 1:]  # without the first position <|endoftext|>
x_hat = cache["blocks.7.hook_resid_pre.hook_sae_recons"][0, 1:]

fve = ...  # Your code here: 1 - sum of squared errors / sum of squared deviations of x from its mean over positions
print("fraction of variance explained:", fve)

**Task 13. What fraction of the activation variance does the SAE reconstruction explain on this prompt? Round the answer to two decimal places.**

Finally, let us substitute the SAE reconstruction for the layer-7 activation and see what happens to the prediction. We measure how far the distributions diverge with the KL divergence.

In [ ]:
clean_logits = model(prompt)[0, -1]
spliced_logits = model.run_with_saes(prompt, saes=[sae])[0, -1]  # the SAE reconstruction replaces the layer-7 activation

for name, logits in [("without SAE", clean_logits), ("with SAE reconstruction", spliced_logits)]:
    print(name, [model.tokenizer.decode(t) for t in logits.topk(3).indices])

p, q = clean_logits.softmax(-1), spliced_logits.softmax(-1)
print("KL(without SAE || with SAE):", round((p * (p.log() - q.log())).sum().item(), 3))

**Task 14. What changed in the model's prediction when the layer-7 activation was replaced with the SAE reconstruction?**

## **Conclusion**

- The toy model shows superposition in its pure form: with dense features the model stores as many features as it has dimensions and drops the rest; with sparse features it packs more features than dimensions into nearly orthogonal directions and suppresses the interference with a negative bias.
- An SAE tries to unpack this packing: the dictionary is 32 times wider than the activation, and a few dozen features are non-zero at one token.
- The SAE reconstruction is not exact: it does not explain all of the activation variance and changes the model's prediction a little. So the conclusion «the model uses this feature» requires an intervention, not only reading the feature.
- Feature descriptions are hypotheses of an annotator model; they are checked on new texts.